# Exportación del Modelo Relacional a Excel

**Propósito:** leer las 8 tablas del modelo relacional desde `database/telco.db`  
y exportarlas a un Excel con 8 hojas, una por tabla.  
Este archivo es el input para Tableau Public.

**Output:** `data/telco_modelo.xlsx`

## Sección 0 — Configuración

In [1]:
import os
import sqlite3
import pandas as pd
import openpyxl  # requerido por pd.ExcelWriter
from pathlib import Path

In [2]:
# Garantizar que el working directory sea la raiz del proyecto
_cwd = Path.cwd()
if _cwd.name == 'notebooks':
    os.chdir(_cwd.parent)
print('Working directory verificado')

PATH_DB     = 'database/telco.db'
PATH_OUTPUT = 'data/telco_modelo.xlsx'

conn = sqlite3.connect(PATH_DB)
conn.execute('PRAGMA foreign_keys = ON')

Working directory verificado


In [3]:
query_tablas = """
    SELECT name FROM sqlite_master
    WHERE type='table'
      AND name IN (
        'dim_genero','dim_cliente','dim_tipo_contrato',
        'dim_metodo_pago','dim_tipo_internet','dim_catalogo_servicios',
        'bridge_cliente_servicio','fact_cliente'
      )
"""
tablas_encontradas = pd.read_sql(query_tablas, conn)['name'].tolist()

assert len(tablas_encontradas) == 8, (
    f'Se esperaban 8 tablas, se encontraron {len(tablas_encontradas)}: {tablas_encontradas}'
)
print(f'Tablas verificadas: {sorted(tablas_encontradas)}')

Tablas verificadas: ['bridge_cliente_servicio', 'dim_catalogo_servicios', 'dim_cliente', 'dim_genero', 'dim_metodo_pago', 'dim_tipo_contrato', 'dim_tipo_internet', 'fact_cliente']


## Sección 1 — Lectura de tablas

In [4]:
CONSULTAS = {
    'dim_genero':              'SELECT * FROM dim_genero',
    'dim_cliente':             'SELECT * FROM dim_cliente',
    'dim_tipo_contrato':       'SELECT * FROM dim_tipo_contrato',
    'dim_metodo_pago':         'SELECT * FROM dim_metodo_pago',
    'dim_tipo_internet':       'SELECT * FROM dim_tipo_internet',
    'dim_catalogo_servicios':  'SELECT * FROM dim_catalogo_servicios',
    'bridge_cliente_servicio': 'SELECT * FROM bridge_cliente_servicio',
    'fact_cliente':            'SELECT * FROM fact_cliente',
}

dataframes = {}
for nombre, query in CONSULTAS.items():
    df = pd.read_sql(query, conn)
    dataframes[nombre] = df
    print(f'{nombre}  shape={df.shape}')
    display(df.head(3))

dim_genero  shape=(2, 2)


,genero_id,descripcion
0,1,Female
1,2,Male


dim_cliente  shape=(7043, 5)


,cliente_id,genero_id,SeniorCitizen,Partner,Dependents
0,7590-VHVEG,1,0,Yes,No
1,5575-GNVDE,2,0,No,No
2,3668-QPYBK,2,0,No,No


dim_tipo_contrato  shape=(3, 2)


...

,servicio_id,nombre,categoria
0,1,PhoneService,Telefonia
1,2,MultipleLines,Telefonia
2,3,OnlineSecurity,Internet


bridge_cliente_servicio  shape=(56344, 3)


,cliente_id,servicio_id,estado
0,7590-VHVEG,1,No
1,5575-GNVDE,1,Yes
2,3668-QPYBK,1,Yes


fact_cliente  shape=(7043, 15)


,cliente_id,tipo_contrato_id,metodo_pago_id,tipo_internet_id,tenure,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,Churn_num,tenure_group,perfil_familiar,n_servicios,rango_cargo,segmento_riesgo
0,7590-VHVEG,1,3,1,1,Yes,29.85,29.85,No,0,01_0-12 meses,"Con pareja, sin dependientes",1,01_Bajo (<=35),0
1,5575-GNVDE,2,4,1,34,No,56.95,1889.50,No,0,03_25-48 meses,Sin pareja ni dependientes,2,02_Medio (35-65),0
2,3668-QPYBK,1,4,1,2,Yes,53.85,108.15,Yes,1,01_0-12 meses,Sin pareja ni dependientes,2,02_Medio (35-65),0


In [5]:
df_dim_genero             = dataframes['dim_genero']
df_dim_cliente            = dataframes['dim_cliente']
df_dim_tipo_contrato      = dataframes['dim_tipo_contrato']
df_dim_metodo_pago        = dataframes['dim_metodo_pago']
df_dim_tipo_internet      = dataframes['dim_tipo_internet']
df_dim_catalogo_servicios = dataframes['dim_catalogo_servicios']
df_bridge                 = dataframes['bridge_cliente_servicio']
df_fact                   = dataframes['fact_cliente']

In [6]:
resumen = pd.DataFrame(
    [{'Tabla': nombre, 'Filas': df.shape[0], 'Columnas': df.shape[1]}
     for nombre, df in dataframes.items()]
)
print('Resumen del modelo relacional:')
display(resumen)

Resumen del modelo relacional:


,Tabla,Filas,Columnas
0,dim_genero,2,2
1,dim_cliente,7043,5
2,dim_tipo_contrato,3,2
3,dim_metodo_pago,4,2
4,dim_tipo_internet,3,2
5,dim_catalogo_servicios,8,3
6,bridge_cliente_servicio,56344,3
7,fact_cliente,7043,15


## Sección 2 — Exportación a Excel

In [7]:
with pd.ExcelWriter(PATH_OUTPUT, engine='openpyxl') as writer:
    df_dim_genero.to_excel(writer,
        sheet_name='dim_genero', index=False)
    df_dim_cliente.to_excel(writer,
        sheet_name='dim_cliente', index=False)
    df_dim_tipo_contrato.to_excel(writer,
        sheet_name='dim_tipo_contrato', index=False)
    df_dim_metodo_pago.to_excel(writer,
        sheet_name='dim_metodo_pago', index=False)
    df_dim_tipo_internet.to_excel(writer,
        sheet_name='dim_tipo_internet', index=False)
    df_dim_catalogo_servicios.to_excel(writer,
        sheet_name='dim_catalogo_servicios', index=False)
    df_bridge.to_excel(writer,
        sheet_name='bridge_cliente_servicio', index=False)
    df_fact.to_excel(writer,
        sheet_name='fact_cliente', index=False)

print(f'Archivo exportado: {PATH_OUTPUT}')

Archivo exportado: data/telco_modelo.xlsx


## Sección 3 — Verificación

In [8]:
FILAS_ESPERADAS = {
    'dim_genero':              2,
    'dim_tipo_contrato':       3,
    'dim_metodo_pago':         4,
    'dim_tipo_internet':       3,
    'dim_catalogo_servicios':  8,
    'dim_cliente':             7043,
    'fact_cliente':            7043,
    'bridge_cliente_servicio': 56344,
}

xl = pd.ExcelFile(PATH_OUTPUT)

# Verificar número de hojas
assert len(xl.sheet_names) == 8, (
    f'Se esperaban 8 hojas, se encontraron {len(xl.sheet_names)}'
)

# Verificar nombres exactos
hojas_esperadas   = set(FILAS_ESPERADAS.keys())
hojas_encontradas = set(xl.sheet_names)
assert hojas_encontradas == hojas_esperadas, (
    f'Nombres incorrectos. Esperadas: {sorted(hojas_esperadas)} '
    f'Encontradas: {sorted(hojas_encontradas)}'
)

# Verificar conteo de filas
for hoja, filas_esp in FILAS_ESPERADAS.items():
    df_ver     = xl.parse(hoja)
    filas_real = len(df_ver)
    estado     = 'OK  ' if filas_real == filas_esp else 'FAIL'
    print(f'{estado}  {hoja:<28} esperadas={filas_esp:>6}  encontradas={filas_real:>6}')

# Tamaño del archivo
tam_mb = Path(PATH_OUTPUT).stat().st_size / (1024 * 1024)
print(f'\nTamaño del archivo: {tam_mb:.2f} MB')

conn.close()

OK    dim_genero                   esperadas=     2  encontradas=     2
OK    dim_tipo_contrato            esperadas=     3  encontradas=     3
OK    dim_metodo_pago              esperadas=     4  encontradas=     4
OK    dim_tipo_internet            esperadas=     3  encontradas=     3
OK    dim_catalogo_servicios       esperadas=     8  encontradas=     8


OK    dim_cliente                  esperadas=  7043  encontradas=  7043


OK    fact_cliente                 esperadas=  7043  encontradas=  7043


OK    bridge_cliente_servicio      esperadas= 56344  encontradas= 56344

Tamaño del archivo: 1.77 MB


## Resultado

- **Archivo generado:** `data/telco_modelo.xlsx`
- **Hojas:** 8 (una por tabla del modelo relacional)
- **Uso:** conectar en Tableau Public como fuente de datos
- **Próximo paso:** definir relaciones en Tableau igual que Power BI